# From GLMs to model-based boosting

This chapter develops model-based boosting from three earlier ideas: generalized
linear models, stagewise residual correction, and weak-to-strong learning. It
explains why the algorithm updates one component at a time, shrinks each update,
and uses early stopping. Mathematical details are derived in the chapters linked
throughout.

```{admonition} Roadmap
:class: tip
An unpenalized GLM estimates all coefficients jointly. With more coefficients than
independent observations, the coefficients are not uniquely identified; strong
collinearity makes them poorly determined even when a unique estimate exists.
Model-based boosting instead follows a regularized path. At each iteration it
computes the loss's negative gradient, fits every candidate base learner to that
pseudo-response, selects one learner, and adds a shrunken version of its fit.
The candidate learners define the permitted effect shapes; the loss defines the
response feature being estimated; early stopping controls path complexity.
```



## 1. The problem: fitting models with many predictors



### 1.1 Limits of an unpenalized joint fit

Consider $n=200$ patients and $p=10{,}000$ gene-expression measurements. The
linear predictor of a GLM is $\eta=X\beta$. Because
$\operatorname{rank}(X)\leq n<p$, many coefficient vectors produce the same
$\eta$; the unpenalized coefficients are therefore not identifiable. When
$p<n$ but columns of $X$ are nearly dependent, a unique estimate may exist but
have high variance: small changes in the data can cause large coefficient changes.

Three common responses impose different structures on the problem:

1. **Penalized regression** (ridge, lasso, elastic net): add a penalty
   $\lambda\|\beta\|_1$ or $\lambda\|\beta\|_2^2$ to make the problem
   well-posed. Lasso ($\ell_1$) also does variable selection.
2. **Dimensionality reduction** (PCA, PLS): project $X$ onto a low-dimensional
   subspace before fitting.
3. **Boosting** (this book): build the predictor incrementally, one component at
   a time, with automatic selection.

Penalization changes the estimation criterion; dimension reduction changes the
predictor representation; boosting changes the fitting path. In model-based
boosting, each candidate learner represents one effect—such as a linear term,
smooth, tree, or random effect—and the loss supplies the pseudo-response. This
separation permits structured additive models under Gaussian, binary, count,
survival, and quantile losses without jointly estimating every coefficient.



### 1.2 The forward stagewise idea

A direct precursor is **forward stagewise regression**. Start with the null
model; at each step choose the predictor most aligned with the current residual
and update only its coefficient by a small amount. The other coefficients are
not jointly re-estimated.

```{figure} ../figures/mermaid-cdbb14f4883b.svg
:name: mermaid-cdbb14f4883b
:alt: Successive forward-stagewise updates start from the response mean and repeatedly add a shrunken coefficient for the predictor that best explains the current residual.

Forward stagewise regression builds the predictor through small updates to one selected component at a time.
```

If predictor $j$ is selected, the update has the form
$\beta_j^{[m]}=\beta_j^{[m-1]}+\nu\hat\delta_j$. Selecting $j$ again can
increase or decrease its cumulative coefficient; “freezing” means only that
coefficients are not recomputed by a joint refit of the active set (the predictors with nonzero coefficients). Small updates limit
the effect of any one selection and create a path of gradually changing models.

**Forward stepwise regression** instead adds a variable and refits all active
coefficients. That joint refit can make a large move after an early selection and
be unstable when active predictors are strongly correlated. Stagewise regression
takes more iterations, but its step size directly controls how quickly the fit can
adapt to each selected variable.

```{admonition} Stagewise vs stepwise
:class: important

- **Forward stepwise**: after adding $x_j$, refit all active coefficients. The
  active set defines a joint least-squares problem, so adding one variable can
  change every earlier coefficient.

- **Forward stagewise**: update one coefficient and leave all others unchanged in
  that iteration. A later iteration may select the same variable again, including
  with an increment of the opposite sign.

**Boosting is a forward stagewise procedure**: it fits a base learner to the
negative gradient of the current loss, multiplies that fit by the step length
$\nu$, and adds it to the predictor. The selection sequence and its increments
specify the fitted path.
```



### 1.3 Older roots: twicing and greedy residual correction

Before boosting had its modern name, statisticians already used the same
residual-correction instinct. Tukey's *Exploratory Data Analysis* described
"twicing": apply a smoother $S$ to $y$, smooth the leftover residual $y - Sy$,
and add the correction back:

$$
Sy + S(y - Sy) = (2S - S^2)y.
$$

With a fixed linear smoother and squared-error loss, this is exactly two
full-size L2Boosting steps with a single base learner. Modern component-wise
boosting keeps the residual-refitting idea but adds learner selection, shrinkage
$\nu < 1$, and early stopping, turning a one-off bias correction into a
regularized path {cite}`tukey1977exploratory,buhlmann2007boosting`.

There is also a numerical-analysis lineage. For linear least-squares base
learners, choosing the variable that most improves the current residual is a Gauss-Southwell coordinate move (a coordinate-descent step that updates the coordinate with the largest gradient entry); with richer dictionaries (large sets of candidate functions to choose from) it resembles matching
pursuit, a greedy method that adds the candidate most correlated with the current residual
{cite}`southwell1946relaxation,mallat1993matching,temlyakov2000weak`.
Together, these older views clarify component-wise boosting: it is residual
correction, coordinate selection, and regularization by path length, expressed
in statistical-model language.



## 2. The evolution: from weak learning to model-based boosting



### 2.1 Weak learning and AdaBoost

In machine learning, boosting began with the weak-to-strong learning question:
can a rule that is only slightly better than guessing be called repeatedly and
combined into an arbitrarily accurate predictor? Kearns and Valiant introduced weak
learnability and posed this equivalence question in work first presented in 1989
{cite}`kearns1989cryptographic`. Schapire gave the first constructive positive
answer in 1990 {cite}`schapire1990strength`; Freund and Schapire's later
**AdaBoost** algorithm made the idea practical
{cite}`freund1997decision`.

For binary labels $y_i\in\{-1,+1\}$, AdaBoost starts with equal observation
weights. At round $m$ it fits a classifier $h_m$, computes its weighted error
$\epsilon_m$, and assigns it the vote
$\alpha_m=\tfrac12\log((1-\epsilon_m)/\epsilon_m)$. The next weights satisfy
$w_i^{[m+1]}\propto w_i^{[m]}\exp(-\alpha_m y_i h_m(x_i))$, so a mistake is
upweighted and a correct classification is downweighted.

The AdaBoost ensemble is

$$
F_M(x) = \sum_{m=1}^M \alpha_m\, h_m(x),
$$

where $h_m$ are weak classifiers, often decision stumps. If $Z_m$ is the
normalizing constant in the weight update, Freund and Schapire's bound is
$\Pr_{\mathrm{train}}[y_i\neq\operatorname{sign}F_M(x_i)]\leq\prod_m Z_m$.
A uniform positive edge over random guessing makes this upper bound decrease
exponentially with the number of rounds.

```{figure} ../figures/mermaid-0d31b5cf4345.svg
:name: mermaid-0d31b5cf4345
:alt: Three AdaBoost rounds reweight misclassified observations before fitting the next weak classifier, followed by a weighted ensemble vote.

AdaBoost refits a weak classifier after increasing the weights of previously misclassified observations, then combines the classifiers by weighted voting.
```



### 2.2 AdaBoost as gradient descent

Breiman interpreted arcing and boosting as optimization. Friedman, Hastie, and
Tibshirani then derived AdaBoost as **forward stagewise additive modeling** for
the exponential loss $\rho(y,f)=e^{-yf}$, with
$y\in\{-1,+1\}$; Mason and coauthors formulated the broader
functional-gradient view
{cite}`breiman1998arcing,friedman2000additive,mason2000boosting`. Under the
exponential loss, an observation's weight is proportional to $e^{-y_iF(x_i)}$:
the margin used by AdaBoost's weight update. General gradient boosting computes
the negative derivative of the chosen loss at the current predictor and fits a
base learner to that pseudo-response.

This reframing had two consequences:

1. The loss determines the pseudo-residual. Exponential, logistic, squared-error,
   absolute-error, quantile, and Cox partial-likelihood losses require different
   gradients or subgradients but can share the selection-and-update structure.
2. The candidate function class is independent of the loss. Trees, linear terms,
   splines, spatial effects, and random effects can serve as base learners if they
   can be fitted to the pseudo-response under the learner's penalty.



### 2.3 L2Boosting: the Gaussian special case

**L2Boosting** {cite}`buhlmann2003boosting` is boosting with squared-error loss
$\rho(y, f) = \tfrac12(y - f)^2$. The negative gradient is the *residual*
$u_i = y_i - f_i$, so each boosting step fits a base learner to the current
residuals. With one linear learner per predictor, this recovers the stagewise
residual-update mechanism from §1.2.

L2Boosting formalizes the earlier twicing idea by repeating residual
corrections, usually with $\nu < 1$, and using $m_{\text{stop}}$ as the main
regularizer.

```{figure} ../figures/mermaid-247f281a84cc.svg
:name: mermaid-247f281a84cc
:alt: Five-step L2Boosting cycle from the current fit through residual calculation, fitting all learners, minimum-SSE selection, and a shrunken update.

An L2Boosting iteration computes residuals, fits every candidate learner, selects the minimum-SSE fit, and adds a shrunken update.
```

After $m_{\text{stop}}$ iterations: $\hat f = f^{[0]} + \nu \sum_m \hat g_{j^\star_m}$.

Bühlmann and Yu established consistency results for early-stopped L2Boosting in
high-dimensional linear models under conditions on the design, signal, and
growth of the stopping iteration. The stopping iteration determines the fitted estimator and acts as a
regularizer. The assumptions and rates are
discussed in [](theory_properties.ipynb).



### 2.4 Model-based boosting: the general framework

**Model-based boosting** {cite}`buhlmann2007boosting,hothorn2010mboost`
extends L2Boosting by separating the loss from the candidate effects. The loss
must provide a usable negative gradient or subgradient. Each base learner defines
one structured effect—such as a linear term, penalized spline, tree, spatial
surface, or random effect—and a fitting rule for the pseudo-response. R's
`mboost` and `mboostrs` implement this framework.

```{figure} ../figures/mermaid-b7813ef0114a.svg
:name: mermaid-b7813ef0114a
:alt: The family determines the negative gradient, structured base learners fit it by penalized least squares, and shrinkage plus early stopping regularize the path.

Model-based boosting separates the loss, candidate base learners, and path regularization.
```

Changing from Gaussian mean regression to quantile or binary regression changes
the loss, offset, and pseudo-residual. It does not change how candidate learners
are fitted, compared, selected, and added. The result remains an additive sum of
the selected effects, so each update can be traced to one learner.



## 3. Component-wise selection



### 3.1 Candidate fitting and selection

In `mboost`-style **component-wise** boosting, each learner represents one model
component. Every learner is fitted to the current pseudo-residual, and the learner
with the smallest fitting criterion is selected for the update. This differs from
tree-based gradient boosting in where variable selection occurs:

- **Gradient boosting** (Friedman): fits a tree to the gradient and adds the
  whole tree. The tree itself selects variables internally.
- **Component-wise boosting** (mboost): each base learner is a *single effect*
  (one variable, one smooth, one group). Selection happens at the *learner*
  level, not inside the learner.

```{figure} ../figures/mermaid-c80670d12baf.svg
:name: mermaid-c80670d12baf
:alt: Parallel candidate learner fits to the negative gradient produce SSE values; only the minimum-SSE learner contributes the next update.

Component-wise selection fits every candidate effect to the pseudo-residual and updates only the learner with minimum fitting error.
```



### 3.2 Automatic variable selection

A learner that is never selected contributes exactly zero to the fitted additive
predictor. Selection is therefore embedded in fitting rather than applied later by
thresholding coefficients. The selected set depends on the stopping iteration:
running longer gives additional learners opportunities to enter.

```{figure} ../figures/fig-selection-path.svg
:label: fig-selection-path
:alt: Selection path showing which variables are selected across 50 boosting iterations

Which learners enter the model, and when. Selection is a by-product of fitting: a learner never chosen has coefficient exactly zero, and the selected set grows with the stopping iteration.
```

At the stopping iteration shown, $x_3$ and $x_5$ have never been selected, so
their fitted contributions are zero.

This construction can fit when $p\gg n$ because it solves many small learner
problems instead of one unidentified $p$-coefficient problem. Early stopping can
keep the selected model sparse, although sparsity is not guaranteed if the chosen
stopping iteration is large.



### 3.3 Learner flexibility and selection bias

Selection compares in-sample fits to the same pseudo-response. A more flexible
learner can reduce that criterion by fitting noise, so it has more opportunities to
win even when its underlying covariate carries no more signal. Penalized learners
therefore use small, comparable effective degrees of freedom whenever the effect
types permit it; `df=4` is the default for many smooth learners.

```{list-table}
:header-rows: 1

* - Setting
  - bbs(x₁)
  - bbs(x₂)
  - bbs(x₃)
  - Effect
* - **Unfair** (different df)
  - df=4
  - **df=20**
  - df=4
  - Selection is biased toward x₂'s flexibility
* - **Comparable** (all df=4)
  - df=4
  - df=4
  - df=4
  - Removes this source of flexibility bias
```

Equal degrees of freedom do not guarantee correct selection: correlated
covariates, heterogeneous penalties, and random variation still matter. They do
remove a direct advantage caused by nominal model size. The mechanics of the
`df`-to-$\lambda$ mapping are in
[](penalties_df.ipynb); the practical guidance is in [](base_learners.ipynb).



## 4. Early stopping as regularization



### 4.1 Regularization along the boosting path

At $m_{\text{stop}}=0$, the predictor contains only the offset. Each additional
iteration adds one shrunken learner fit, enlarging the path traversed through the
candidate function space. If run sufficiently long, the procedure approaches an
unregularized fit when such a limit exists. Thus $m_{\text{stop}}$ is the primary
regularization parameter. Training risk is optimized along the path and normally
decreases. Held-out risk often decreases initially and later increases, but its
finite-sample curve need not be smooth or exactly U-shaped:

```{figure} ../figures/fig-boosting-risk-curve.svg
:label: fig-risk-curve
:alt: Training and test risk as a function of m_stop

Training risk falls monotonically along the path while held-out risk turns up once the fit begins tracking noise. The held-out curve is not guaranteed to be smooth or exactly U-shaped in finite samples.
```



### 4.2 The mechanism of early stopping

Early stopping limits the cumulative sequence of data-adaptive updates. With a
small step length, representing a complex or weak effect generally requires many
selections, whereas a learner aligned with a large current gradient can reduce the
loss early. Truncating the path can therefore suppress later, smaller adjustments.
This is a tendency, not an ordering theorem: noise variables can enter early, and
correlated learners can exchange selections. Held-out risk estimates determine
whether the additional adjustments improve prediction.

The comparison with lasso is structural. Decreasing the lasso penalty $\lambda$
relaxes a global $\ell_1$ constraint; increasing $m_{\text{stop}}$ permits more
sequential boosting updates. Both generate paths from more to less regularized
fits, but their paths solve different optimization problems and need not select
the same variables.

```{admonition} Boosting vs Lasso: a key structural difference
:class: important

| Property | Lasso ($\ell_1$) | Boosting |
|---|---|---|
| Path construction | solve a penalized objective for each $\lambda$ | append one selected update |
| Regularizer | $\lambda$ (continuous) | $m_{\text{stop}}$ (integer) |
| Zero effects | induced by the $\ell_1$ penalty | learners not selected before stopping |
| Nonlinear effects | represented by supplied basis columns | represented by a base learner (`bbs`, `bmono`, ...) |
| Non-Gaussian loss | requires a loss-specific penalized solver | requires its gradient or subgradient |
| Correlated predictors | coefficients depend on the joint penalized solution | selections depend on the current residual |

A lasso fit is attractive when the model is linear and a convex penalized
objective is available. Model-based boosting is useful when selection must range
over heterogeneous structured effects or the desired loss is already expressed
through the boosting family interface.
```



### 4.3 Choosing $m_{\text{stop}}$

Cross-validation estimates risk along a grid of stopping iterations. For each split, `cvrisk` (the cross-validation function of the R package `mboost`) fits on the training rows and evaluates the specified loss on the held-out rows at every grid point. It then averages risk across splits
and selects the minimizing iteration $\hat m$. See [](model_selection.ipynb) for
the fold construction, loss scale, uncertainty, and selection rules.

Training risk cannot estimate generalization risk because the same observations
choose every learner and increment. Cross-validated risk may be noisy or have
several near-minima; the selected $\hat m$ is therefore an estimate whose
stability should be inspected, not a known population optimum.



## 5. Extending to distributional regression: GAMLSS via boosting



### 5.1 Distribution parameters beyond the mean

All of the above models a **single** aspect of the response: the conditional
mean (Gaussian), median (Laplace), probability (Binomial), rate (Poisson), or
quantile. But real distributions have **multiple** parameters:

```{list-table}
:header-rows: 1

* - Distribution
  - Parameters
  - What GLM/GAM models
  - What GAMLSS models
* - Gaussian
  - $\mu, \sigma^2$
  - $\mu(X)$ only; $\sigma^2$ constant
  - $\mu(X)$ **and** $\sigma(X)$
* - Gamma
  - $\mu, \sigma$
  - $\mu(X)$ only; $\sigma$ constant
  - $\mu(X)$ **and** $\sigma(X)$
* - Student-t
  - $\mu, \sigma, \nu$
  - $\mu(X)$ only; $\sigma, \nu$ constant
  - $\mu(X)$, $\sigma(X)$, **and** $\nu(X)$
* - NBinomial
  - $\mu, \sigma$
  - $\mu(X)$ only; $\sigma$ constant
  - $\mu(X)$ **and** $\sigma(X)$
```

**GAMLSS** (Generalized Additive Models for Location, Scale and Shape)
{cite}`rigby2005generalized` assigns a predictor to each modeled distribution
parameter. Covariates may therefore affect dispersion or shape even when their
location effect is small, and prediction intervals may change width with the
covariates rather than assuming constant conditional variance.



### 5.2 Boosting GAMLSS: `gamboostLSS`

Mayr and coauthors extended model-based boosting to GAMLSS
{cite}`mayr2012generalized`. Parameter $\theta_k$ is linked to its own additive
predictor $\eta_k$ and candidate learners. With joint negative log-likelihood
$\ell(\eta_1,\ldots,\eta_K)$ as the loss, its pseudo-response is
$u_k=-\partial\ell/\partial\eta_k$. Learner selection can consequently differ
by parameter: a covariate may explain location without explaining scale.

```{figure} ../figures/mermaid-409a1c730568.svg
:name: mermaid-409a1c730568
:alt: A joint negative log-likelihood supplies parameter-specific gradients to separate location and scale learner sets and predictors.

Distributional boosting maintains separate additive predictors and learner-selection paths for each distribution parameter while optimizing a joint risk.
```

A **cyclic** schedule visits parameters in a fixed order and gives each one a
separate stopping count. A **non-cyclic** schedule evaluates candidate updates
across parameters and applies the one with the largest joint-risk reduction, using
one global stopping count. [](gamlss_framework.ipynb) defines the distributional
model; [](gamlss.ipynb) derives both boosting schedules.



## 6. Relation to other regularized learners

```{figure} ../figures/mermaid-5ca15c1707cb.svg
:name: mermaid-5ca15c1707cb
:alt: Comparison of model-based boosting with lasso or elastic net, random forests, and tree gradient boosting.

Model-based boosting, penalized regression, random forests, and tree boosting differ in their candidate components and regularization mechanisms.
```

| Method | Candidate component | Regularization | Interactions |
|---|---|---|---|
| Model-based boosting | one specified additive effect | shrinkage and early stopping | only when represented by a learner |
| Tree gradient boosting | one fitted tree | depth, shrinkage, and early stopping | discovered through tree splits |
| Lasso | one design-matrix coefficient | global $\ell_1$ penalty | only columns included in the design |
| Random forest | one randomized tree | averaging and tree constraints | discovered through tree splits |

The appropriate comparison depends on the estimand and function class.
Model-based boosting is designed for a prespecified additive decomposition that
may mix linear, smooth, monotone, spatial, and random effects. Tree ensembles are
better aligned with problems where unspecified interactions are central. Lasso is
a natural baseline when the design matrix is fixed and a convex sparse linear fit
matches the scientific question. Predictive performance among them is empirical;
none is uniformly preferable.


## 7. Numerical illustration of path behavior

For a full-rank linear Gaussian example, the following calculation illustrates
two properties of the path: a sufficiently long L2Boosting fit approaches OLS,
and the iteration count required to reach a fixed risk scales approximately with
$1/\nu$ when the step length is small. The calculation is an example, not a proof
of the convergence results cited in §2.3.


In [1]:
import numpy as np
import mboostrs as mb

rng = np.random.default_rng(0)
n, p = 200, 5
X = rng.normal(size=(n, p))
beta_true = np.array([1.5, -2.0, 0.0, 0.8, 0.0])
y = X @ beta_true + rng.normal(size=n) * 0.5
data = {"y": y, **{f"x{j}": X[:, j] for j in range(p)}}
formula = "y ~ " + " + ".join(f"bols(x{j})" for j in range(p))

# Long-run L2Boosting and OLS on this full-rank design.
ols = np.linalg.lstsq(np.column_stack([np.ones(n), X]), y, rcond=None)[0]
fit = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                   control=mb.boost_control(mstop=5000, nu=0.1))
boosted = fit.coef(off2int=True)
boosted_vec = np.array([boosted["(Intercept)"]] + [boosted[f"x{j}"] for j in range(p)])
print("OLS coefficients     :", np.round(ols, 4))
print("boosting (mstop=5000):", np.round(boosted_vec, 4))
np.testing.assert_allclose(boosted_vec, ols, atol=1e-3)

# Iterations required to reach a fixed risk at three step lengths.
reference = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                         control=mb.boost_control(mstop=2000, nu=0.2))
target_risk = reference.risk_[50]  # risk reached at m=50 with nu=0.2
print(f"\ntarget risk (nu=0.2, m=50): {target_risk:.4f}")
for nu in [0.2, 0.1, 0.05]:
    fit_nu = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                         control=mb.boost_control(mstop=2000, nu=nu))
    m_needed = int(np.argmax(fit_nu.risk_ <= target_risk))
    print(f"  nu={nu:<5}: mstop to reach that risk = {m_needed}")


OLS coefficients     : [ 0.0219  1.5418 -2.0397 -0.0164  0.8235  0.0509]
boosting (mstop=5000): [ 0.0219  1.5418 -2.0397 -0.0164  0.8235  0.0509]

target risk (nu=0.2, m=50): 48.2493
  nu=0.2  : mstop to reach that risk = 50
  nu=0.1  : mstop to reach that risk = 106
  nu=0.05 : mstop to reach that risk = 217


On this data, the long boosting path agrees with OLS to three decimal places.
Halving $\nu$ approximately doubles the iterations required to reach the chosen
risk, consistent with comparing paths by the accumulated step length
$t=m\nu$. Finite step sizes, learner switches, and ties prevent this scaling from
being an exact identity.


## 8. Summary

- A loss supplies the pseudo-residual; base learners supply candidate effect
  shapes. One fitted learner is selected and added at each iteration.
- Learners not selected before $m_{\text{stop}}$ contribute zero. Comparable
  effective degrees of freedom reduce, but do not eliminate, selection bias.
- The step length $\nu$ controls each update, while $m_{\text{stop}}$ limits the
  cumulative path. Cross-validated risk estimates where to stop.
- Distributional boosting repeats this construction for multiple linked
  parameters and selects effects separately for location, scale, or shape.

## 9. Detailed treatments

- [](functional_gradient_boosting.ipynb) derives the algorithm and Gaussian hat
  matrix. [](base_learners.ipynb) and [](penalties_df.ipynb) derive learner fits
  and effective-degree-of-freedom calibration.
- [](families.ipynb) defines losses, gradients, and offsets.
  [](gamlss_framework.ipynb) and [](gamlss.ipynb) extend them to multi-parameter
  distributions.
- [](theory_properties.ipynb), [](model_selection.ipynb), and
  [](stability_selection.ipynb) cover convergence, stopping, and selection error.

## See also

- [](../usage/glmboost) applies the linear interface.
- [](../implementation/architecture) and [](../implementation/engine) map the
  statistical components to the Rust types and boosting loop.

```{bibliography}
:filter: docname in docnames
```